# GaiaCMD — Example Usage

This notebook demonstrates how to use the `gaiacmd` package to query Gaia EDR3
and plot a Color-Magnitude Diagram for a catalog of TESS white dwarf targets.

## What this package does

1. Reads a CSV file of TIC IDs
2. Looks up RA/Dec for each target from the TIC catalog
3. Queries Gaia EDR3 for each target and a surrounding 1° field
4. Plots all targets on a Gaia CMD with a background field star cloud

## Install

```bash
pip install gaiacmd
```

In [ ]:
from gaiacmd import run_cmd, plot_cmd

## Input format

Your CSV file needs at minimum a `TIC` column:

```
TIC
141872267
123456789
987654321
```

It can have other columns too — they will be ignored for now.

## Step 1 — Run the Gaia queries

This queries Gaia EDR3 for every TIC in your CSV.

- Progress is saved to a checkpoint file after every single TIC
- If interrupted, just re-run this cell — it picks up where it left off
- Delete `gaia_cmd_checkpoint.pkl` to start completely fresh

In [ ]:
state = run_cmd(
    csv_file         = "my_tics.csv",           # path to your TIC list
    tic_column       = "TIC",                   # column name in the CSV
    log_file         = "skipped_tics.log",      # skipped objects are logged here
    checkpoint_file  = "gaia_cmd_checkpoint.pkl",
    field_radius_deg = 1.0,                     # Gaia cone search radius (degrees)
    max_match_arcsec = 10.0,                    # max TIC-Gaia separation to accept
)

## Step 2 — Plot the CMD

Once queries are done, plot instantly from the collected state.
You can re-run this cell with different settings without re-querying Gaia.

In [ ]:
fig, ax = plot_cmd(
    state,
    output_plot = "gaia_cmd_wds.png",   # set to None to skip saving
    title       = "Gaia CMD — TESS WD Pulsation Catalog",
    xlim        = (-1.0, 3.0),
    ylim        = (18.0, 0.0),          # inverted: bright at top
)

## What's in `state`?

The `state` dict returned by `run_cmd()` contains everything collected:

| Key | Contents |
|---|---|
| `wd_colors` | BP-RP colors of your WD targets |
| `wd_mgs` | Absolute G magnitudes of your WD targets |
| `wd_labels` | TIC IDs (same order as above) |
| `field_colors` | BP-RP colors of background field stars |
| `field_mgs` | Absolute G magnitudes of field stars |
| `skipped` | List of (TIC_ID, reason) for skipped objects |
| `done_tics` | Set of all processed TIC IDs |

In [ ]:
# Quick summary
print(f"WDs plotted:      {len(state['wd_colors'])}")
print(f"Field stars:      {len(state['field_colors']):,}")
print(f"Skipped:          {len(state['skipped'])}")
print(f"Total processed:  {len(state['done_tics'])}")

## CLI alternative

You can also run the full pipeline from the terminal without any Python:

```bash
gaiacmd --csv my_tics.csv
```

With options:
```bash
gaiacmd --csv my_tics.csv --output my_cmd.png --radius 1.5 --tic-column TIC
```